In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import threading
import copy
import json
import re

import torch
import matplotlib.pyplot as plt
from pathlib import Path
from joblib import Parallel, delayed, load
from torch_geometric.data import Data
from torch_geometric.explain import Explainer, GNNExplainer
from tqdm.auto import tqdm
from sklearn.preprocessing import minmax_scale

import nibabel as nib
from nilearn import plotting

In [ ]:
# Add QuantNets to Path to import your custom architecture
base_dir = Path('/scratch/bng/cartbind/code/MIND_models')
quantnets_path = base_dir / "QuantNets"
if str(quantnets_path) not in sys.path:
    sys.path.append(str(quantnets_path))

from gnn.architectures_inject import GATv2ConvNet as GATv2ConvNet_Inject
from gnn.architectures_normal import GATv2ConvNet as GATv2ConvNet_Normal

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Base directories where your Optuna run saved everything
gnn_dir = base_dir / 'models_gnn_dnanexus8'
predictions_dir = gnn_dir / 'gnn_predictions_scaling_law'
params_dir = gnn_dir / 'gnn_best_params_scaling_law'
weights_dir = gnn_dir / 'gnn_weights_scaling_law'
preprocessors_dir = gnn_dir / 'gnn_preprocessors_scaling_law'
data_dir = Path('/scratch/bng/cartbind/data/UKB_new_data/combined_data_no_outliers')
region_dir = base_dir / 'region_names'

# Reusing your region parsing logic
def parse_region_map(region_file_path):
    with open(region_file_path, 'r') as f:
        lines = [line.strip() for line in f if line.strip()]
    raw_pairs = []
    for line in lines:
        if line.startswith("FC"):
            inner = line[line.find("(")+1:line.find(")")]
            src, dst = inner.split("-")
        else:
            src, dst = line.split("-")
        raw_pairs.append((src, dst))

    unique_nodes = []
    for src, dst in raw_pairs:
        if src not in unique_nodes: unique_nodes.append(src)
        if dst not in unique_nodes: unique_nodes.append(dst)
        
    node_to_idx = {name: idx for idx, name in enumerate(unique_nodes)}
    edge_list = [(node_to_idx[src], node_to_idx[dst]) for src, dst in raw_pairs]
    return len(unique_nodes), edge_list, lines, unique_nodes

In [ ]:
# Wrapper so PyG Explainer can pass arguments cleanly to your model
class GNNModelWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        
    def forward(self, x, edge_index, edge_attr=None, demographics=None, batch=None):
        if batch is None:
            batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)
            
        # EXPLAINER FIX: PyG explainer requires a 2D `x` to avoid crashing, 
        # but your model's Embedding layer strictly requires 1D standard indices.
        # We manually reconstruct the 1D indices right before passing it to your model.
        real_x_node = torch.arange(x.size(0), dtype=torch.long, device=x.device)
        
        # Reconstruct the PyG Data object your architecture expects
        data = Data(x=real_x_node, edge_index=edge_index, edge_attr=edge_attr, demographics=demographics, batch=batch)
        return self.model(data)

def load_trained_environment(target_name, data_name, sample_size):
    """Rebuilds the model, preprocessors, and test predictions for a specific configuration."""
    
    # 1. Load Parameters
    params_path = params_dir / target_name / f'GNN_best_params_{data_name}_{target_name}_{sample_size}.csv'
    params_df = pd.read_csv(params_path)
    p = params_df.iloc[0].to_dict()
    
    # Reconstruct parameter math from optuna pipeline
    hidden_channels = 2 ** int(p['hidden_channels'])
    embedding_dim = 2 ** int(p['embedding_dim'])
    hidden_heads = 2 ** int(p['hidden_heads'])
    demo_embed_dim = 2 ** int(p['demo_embed_dim'])
    classifier_num_layers = int(p['classifier_num_layers'])
    classifier_hidden_dims = [2 ** int(p[f'classifier_dim_exp_l{i}']) for i in range(classifier_num_layers)]
    
    # 2. Get Region Data Structures
    region_file = region_dir / f'{data_name}_regions.txt'
    num_nodes, mapping_edges, brain_cols, unique_nodes = parse_region_map(region_file)

    # 3. Instantiate the exact model
    ModelClass = GATv2ConvNet_Inject if p['architecture'] == 'inject' else GATv2ConvNet_Normal
    
    # We must load preprocessors to find demo_dim (number of demographic columns)
    preprocessors_path = preprocessors_dir / target_name / f'preprocessors_{data_name}_{target_name}_{sample_size}.pkl'
    preprocessors = load(preprocessors_path)
    # The demographic scaler output features size:
    demo_dim = len(preprocessors['demo'].get_feature_names_out())

    raw_model = ModelClass(
        out_dim=1, 
        in_channels=num_nodes,          
        gnn_num_layers=int(p['gnn_num_layers']), 
        classifier_hidden_dims=classifier_hidden_dims,
        hidden_channels=hidden_channels, 
        embedding_dim=embedding_dim,
        demo_dim=demo_dim,
        edge_dropout_rate=float(p['edge_dropout']),
        node_dropout_rate=float(p['node_dropout']),
        hidden_heads=hidden_heads,
        demo_embed_dim=demo_embed_dim,
        include_demo=True,
        classifier_dropout_rate=float(p['classifier_dropout']),
        concat=bool(p['concat']),
        include_jk=bool(p['include_jk'])
    )

    # 4. Load Weights and wrap
    weights_path = weights_dir / target_name / f'GNN_weights_{data_name}_{target_name}_{sample_size}.pt'
    
    # Load state dict and remove the '_orig_mod.' prefix added by torch.compile()
    state_dict = torch.load(weights_path, map_location=device)
    clean_state_dict = {k.replace('_orig_mod.', ''): v for k, v in state_dict.items()}
    
    raw_model.load_state_dict(clean_state_dict)
    raw_model.eval()
    raw_model.to(device)
    
    wrapped_model = GNNModelWrapper(raw_model)
    
    # 5. Load the test predictions (to fetch patient eids)
    preds_path = predictions_dir / target_name / f'GNN_preds_{data_name}_{target_name}_{sample_size}.csv'
    preds_df = pd.read_csv(preds_path)
    
    return wrapped_model, preprocessors, preds_df, mapping_edges, brain_cols, unique_nodes, num_nodes

In [ ]:
# --- Select Your Run Context ---
TARGET_NAME = 'GF'
DATA_NAME = 'MIND'
SAMPLE_SIZE = 'all'

# Load mapping and models
model, preps, test_predictions, mapping_edges, brain_cols, unique_nodes, num_nodes = load_trained_environment(TARGET_NAME, DATA_NAME, SAMPLE_SIZE)

# Fetch the raw dataset to reconstruct exact patient inputs
raw_data_path = data_dir / f'combined_data_master_no_outliers.csv'  # Modify if target changes appropriately
full_df = pd.read_csv(raw_data_path).set_index('eid')

# # Let's say we want to explain the patient with the highest predicted score:
# patient_to_explain = test_predictions.sort_values(by='predicted', ascending=False).iloc[0]
# target_eid = int(patient_to_explain['eid'])

# print(f"Target Patient EID: {target_eid}")
# print(f"Actual {TARGET_NAME}: {patient_to_explain['actual']:.3f} | Predicted {TARGET_NAME}: {patient_to_explain['predicted']:.3f}")

In [ ]:
# # Extract the single patient's raw arrays
# patient_data = pd.DataFrame([full_df.loc[target_eid]])

# # Transform the inputs using the exact same preprocessors used in training
# demo_scaled = torch.tensor(preps['demo'].transform(patient_data), dtype=torch.float32).to(device)
# edges_scaled = preps['edge'].transform(patient_data[brain_cols]).flatten()

# Reconstruct Edges exactly how BrainGraphDataset did
src_global = np.array([e[0] for e in mapping_edges])
dst_global = np.array([e[1] for e in mapping_edges])

# Create undirected graph layout
edge_index_np = np.hstack([np.vstack([src_global, dst_global]), np.vstack([dst_global, src_global])])
edge_index = torch.tensor(edge_index_np, dtype=torch.long).to(device)

# # Map edge weights
# w = np.tile(edges_scaled, 2)
# edge_attr = torch.tensor(w, dtype=torch.float32).unsqueeze(-1).to(device)

# EXPLAINER FIX: Create a 2D dummy tensor to satisfy PyG's internal `N, F = x.size()` unpacking
dummy_x = torch.zeros((num_nodes, 1), dtype=torch.float32).to(device)

# # --- Initialize GNNExplainer ---
# explainer = Explainer(
#     model=model,
#     algorithm=GNNExplainer(epochs=200),
#     explanation_type='model',
#     node_mask_type=None,           
#     edge_mask_type='object',       # Find important edges
#     model_config=dict(
#         mode='regression',
#         task_level='graph',
#         return_type='raw',
#     ),
# )

# print(f"Running explainer (this usually takes a few seconds)...")
# explanation = explainer(
#     x=dummy_x,                     # Pass the 2D dummy variable here
#     edge_index=edge_index, 
#     edge_attr=edge_attr,
#     demographics=demo_scaled,
# )

# print("Explanation generated successfully!")

In [ ]:
# # The Explainer returns masks for the duplicated undirected edge setup
# # We need to average the importance of (A->B) and (B->A) to get a master importance score
# mask_array = explanation.edge_mask.cpu().numpy()
# half_idx = len(mask_array) // 2

# # Average importance of forward and backward direction
# edge_importances = (mask_array[:half_idx] + mask_array[half_idx:]) / 2

# # Map back to string definitions 
# important_links = []
# for i, imp in enumerate(edge_importances):
#     src_idx = mapping_edges[i][0]
#     dst_idx = mapping_edges[i][1]
#     src_name = unique_nodes[src_idx]
#     dst_name = unique_nodes[dst_idx]
    
#     # Store the actual underlying edge weight (from model input) and its importance mask (from GNNExplainer)
#     actual_edge_weight = edges_scaled[i]
#     important_links.append((src_name, dst_name, imp, actual_edge_weight))

# # Sort by most important to the model's decision
# important_links.sort(key=lambda x: x[2], reverse=True)

# print("Top 15 Most Important Functional Connections for this Patient's Prediction:")
# for link in important_links[:15]:
#     print(f"{link[0]} - {link[1]}: Importance = {link[2]:.4f} (Underlying weight: {link[3]:.4f})")

# # --- PLOT AS CIRCLE ---
# TOP_N_EDGES_TO_PLOT = 15

# # Load network groupings for coloring and ordering based on DATA_NAME (e.g., 'FC100')
# json_path = region_dir / f"{DATA_NAME}_network_groupings.json"
# custom_order = []
# ic_to_color = {}

# if json_path.exists():
#     with open(json_path, "r") as f:
#         network_groupings = json.load(f)
#         for network_name, data in network_groupings.items():
#             color = data["color"]
#             for node in data["nodes"]:
#                 custom_order.append(node)
#                 ic_to_color[node] = color

# # Retrieve regions for plotting and apply custom order
# node_names = [n for n in custom_order if n in unique_nodes]
# missing = [n for n in unique_nodes if n not in custom_order]

# # Sort any remaining modules missing from the JSON numerically (if ICxx), otherwise alphabetically
# try:
#     missing.sort(key=lambda x: int(x[2:]))
# except ValueError:
#     missing.sort()

# node_names.extend(missing)
# n_nodes = len(node_names)

# # Map colors to the final order
# node_colors = [ic_to_color.get(name, "gray") for name in node_names]

# node_idx = {name: i for i, name in enumerate(node_names)}
# con_matrix = np.zeros((n_nodes, n_nodes))

# # Fill matrix with importance masks 
# top_edges = important_links[:TOP_N_EDGES_TO_PLOT]
# for n1, n2, imp, _ in top_edges:
#     i, j = node_idx[n1], node_idx[n2]
#     con_matrix[i, j] = imp
#     con_matrix[j, i] = imp

# max_val = np.max(np.abs(con_matrix))
# fig, ax = plt.subplots(figsize=(10, 10), subplot_kw={'projection': 'polar'})

# plot_connectivity_circle(
#     con_matrix,
#     node_names,
#     n_lines=TOP_N_EDGES_TO_PLOT,                  
#     node_colors=node_colors,      # Applied custom colors!
#     colormap="Reds", 
#     vmin=0,               
#     vmax=max_val,                
#     colorbar=True,
#     title=f"GNNExplainer Importance\nPatient: {target_eid} | Pred {TARGET_NAME}: {patient_to_explain['predicted']:.3f}",
#     ax=ax,
#     facecolor='white',
#     textcolor='black',
#     show=False                   
# )
# plt.show()

In [ ]:
# ---------------------------------------------------------
# Plotting Function (MIND Brain Map)
# ---------------------------------------------------------
def plot_global_importance_brain_map(edges_list, title, n_lines=15, colormap="Reds"):
    """
    Plots a 3D brain connectivity map using nilearn for GNNExplainer masks.
    edges_list should be a list of tuples: (source_node, target_node, importance_value)
    """
    # Sort edges by magnitude and just keep top n_lines
    edge_list_sorted = sorted(edges_list, key=lambda x: abs(x[2]), reverse=True)
    top_edges = edge_list_sorted[:n_lines]
    
    # 1. Parse FreeSurfer centroids logic
    def parse_freesurfer_centroids(centroids_path):
        region_to_coord = {}
        with open(centroids_path, 'r') as f:
            for line in f:
                if line.startswith('#') or line.strip() == "":
                    continue
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                label = parts[1]
                coords = tuple(map(float, parts[2:5]))
                match = re.match(r'ctx-(lh|rh)-(.*)', label)
                if match:
                    hemi, region = match.groups()
                    key = f"{hemi}_{region}"
                    region_to_coord[key] = coords
        return region_to_coord

    centroids_file = base_dir / 'centroids.txt' # Make sure base_dir points to MIND_models
    region_to_coord = parse_freesurfer_centroids(centroids_file)
    
    # Extract unique node labels
    node_labels = sorted({n for edge in top_edges for n in edge[:2]})
    coords_native = []
    missing = []
    
    for label in node_labels:
        if label in region_to_coord:
            coords_native.append(region_to_coord[label])
        else:
            coords_native.append((float('nan'), float('nan'), float('nan')))
            missing.append(label)
            
    coords_native = np.array(coords_native)
    label_to_idx = {lab: i for i, lab in enumerate(node_labels)}
    n_nodes = len(node_labels)

    if missing:
        print(f"WARNING: The following regions were not found in centroids.txt:\n{missing}")

    # 2. Transform coordinates to MNI space
    def load_talairach_xfm(xfm_path):
        with open(xfm_path) as f:
            lines = f.readlines()
        start = [i for i, l in enumerate(lines) if 'Linear_Transform' in l][0]
        matrix_lines = lines[start+1:start+4]
        matrix = np.array([
            [float(val.strip(';')) for val in line.strip().split()]
            for line in matrix_lines
        ])
        affine = np.eye(4)
        affine[:3, :4] = matrix
        return affine

    talairach_xfm = '/Applications/freesurfer/subjects/bert/mri/transforms/talairach.xfm'
    affine = load_talairach_xfm(talairach_xfm)
    coords_mni = nib.affines.apply_affine(affine, coords_native)

    # 3. Create strictly sized adjacency matrix
    adj_matrix = np.zeros((n_nodes, n_nodes))
    for n1, n2, w in top_edges:
        i, j = label_to_idx[n1], label_to_idx[n2]
        adj_matrix[i, j] = w
        adj_matrix[j, i] = w  # Plot Undirected

    def scale_coords_to_brain(coords, scale=1.0):
        mean = np.mean(coords, axis=0) 
        coords_centered = coords - mean
        coords_scaled = coords_centered * scale
        coords_scaled = coords_scaled + mean
        return coords_scaled

    coords_mni_scaled = scale_coords_to_brain(coords_mni, scale=1.0)
    
    max_abs_val = np.max(np.abs(adj_matrix))
    if max_abs_val == 0: max_abs_val = 1.0 # fallback

    # 4. Plot Brain Map
    fig = plt.figure(figsize=(10, 8))
    
    display = plotting.plot_connectome(
        adj_matrix,
        coords_mni_scaled,
        edge_threshold=0.0,
        node_color='black',
        node_size=50,
        display_mode='ortho',     # Plot all orthographic views 
        title=title,
        figure=fig,
        edge_cmap=colormap,
        edge_vmin=0,              # Starts at 0 for feature importance
        edge_vmax=max_abs_val,
        node_kwargs={'alpha': 0.9}
    )
    
    # Layout and attach standalone colorbar below
    sm = plt.cm.ScalarMappable(cmap=colormap, norm=plt.Normalize(vmin=0, vmax=max_abs_val))
    sm.set_array([])
    cbar_ax = fig.add_axes([0.35, 0.15, 0.3, 0.03])
    cb = plt.colorbar(sm, cax=cbar_ax, orientation='horizontal')
    cb.set_label('Feature Importance Weight', fontsize=12, labelpad=10)
    
    plt.tight_layout(rect=[0, 0.2, 1, 1])
    plt.show()

In [ ]:
# --- Configuration ---
TOP_K = 15
NUM_TO_RUN = len(test_predictions) # Run on all of them
# NUM_TO_RUN = 100
N_JOBS = 4 # CAUTION: Set to 2-4 if using GPU to avoid OOM. Set to -1 if using CPU to use all cores.
# ---------------------

# 1. Select the patients and isolate their IDs
patients_to_run = test_predictions.head(NUM_TO_RUN)
eids_to_run = patients_to_run['eid'].astype(int).tolist()
n_patients = len(eids_to_run)

# =======================================================================
# OPTIMIZATION 1: Pre-process ALL data to bypass the Python GIL in threads
# =======================================================================
print("Pre-processing patient data (Vectorized)...")
subset_df = full_df.loc[eids_to_run]
all_demos_scaled = preps['demo'].transform(subset_df)
all_edges_scaled = preps['edge'].transform(subset_df[brain_cols])

# Store pre-made tensors locally
demo_dict = {}
edge_dict = {}
for i, eid in enumerate(eids_to_run):
    # Shape [1, demo_features] exactly like your original single-row dataframe transform
    demo_dict[eid] = torch.tensor(all_demos_scaled[i:i+1], dtype=torch.float32).to(device)
    
    # Tile and format edge attributes
    w = np.tile(all_edges_scaled[i], 2)
    edge_dict[eid] = torch.tensor(w, dtype=torch.float32).unsqueeze(-1).to(device)

# =======================================================================
# OPTIMIZATION 2: Cache the entire Explainer, not just the model
# =======================================================================
thread_data = threading.local()

def get_thread_local_explainer(base_model):
    """Fetches a cached Explainer for the current thread, creating it only once."""
    if not hasattr(thread_data, 'explainer'):
        local_model = copy.deepcopy(base_model)
        local_model.eval()
        for param in local_model.parameters():
            param.requires_grad = False
            
        # Cache the Explainer object itself to avoid instantiation overhead
        thread_data.explainer = Explainer(
            model=local_model,
            algorithm=GNNExplainer(epochs=200),
            explanation_type='model',
            node_mask_type=None,           
            edge_mask_type='object',       
            model_config=dict(
                mode='regression',
                task_level='graph',
                return_type='raw',
            ),
        )
    return thread_data.explainer

# =======================================================================
# 3. Stripped down isolated worker function
# =======================================================================
def explain_single_patient(target_eid):
    # Fetch pre-made tensors (Instant, zero pandas/sklearn overhead)
    demo_scaled = demo_dict[target_eid]
    edge_attr = edge_dict[target_eid]
    
    local_explainer = get_thread_local_explainer(model)
    
    # Run the local GNNExplainer
    explanation = local_explainer(
        x=dummy_x,                     
        edge_index=edge_index,         
        edge_attr=edge_attr,
        demographics=demo_scaled,
    )
    
    # Extract, average undirected paths, and normalize
    mask_array = explanation.edge_mask.detach().cpu().numpy()
    half_idx = len(mask_array) // 2
    edge_importances = (mask_array[:half_idx] + mask_array[half_idx:]) / 2

    # Delete the large explanation object so the tensors are immediately 
    # released back to PyTorch's Caching Allocator for the next thread
    del explanation
    
    return minmax_scale(edge_importances)

# 4. Execute in parallel
print(f"Running Explainer on {n_patients} patients using {N_JOBS} parallel threads...")
all_importances_norm = Parallel(n_jobs=N_JOBS, backend="threading")(
    delayed(explain_single_patient)(eid) for eid in tqdm(eids_to_run, desc="Explaining")
)

# 5. Compile Global Metrics
all_importances_norm = np.array(all_importances_norm)
mean_norm_importances = np.mean(all_importances_norm, axis=0)

# Calculate Top-K Frequency
top_k_counts = np.zeros(len(mapping_edges))
for patient_mask in all_importances_norm:
    top_k_indices = np.argsort(patient_mask)[-TOP_K:]
    top_k_counts[top_k_indices] += 1

top_k_frequencies = top_k_counts / n_patients 

# Package results into a list of dictionaries
global_results = []
for i in range(len(mapping_edges)):
    src_idx = mapping_edges[i][0]
    dst_idx = mapping_edges[i][1]
    
    global_results.append({
        'source': unique_nodes[src_idx],
        'target': unique_nodes[dst_idx],
        'mean_norm_imp': mean_norm_importances[i],
        'top_k_freq': top_k_frequencies[i]
    })

print(f"\nCompleted {n_patients} parallel explanations!")

In [ ]:
# Create a directory to store the explainer results
explainer_save_dir = gnn_dir / 'gnn_explainer_results' / TARGET_NAME
explainer_save_dir.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# 1. Save Aggregated Global Results (CSV)
# ---------------------------------------------------------
global_results_df = pd.DataFrame(global_results)

# Sort it by top_k_freq by default for easy viewing
global_results_df = global_results_df.sort_values(by='top_k_freq', ascending=False)

csv_path = explainer_save_dir / f'global_importance_{DATA_NAME}_{TARGET_NAME}_{SAMPLE_SIZE}.csv'
global_results_df.to_csv(csv_path, index=False)
print(f"Saved global aggregated results to:\n{csv_path}")

# ---------------------------------------------------------
# 2. Save Patient-Level Importances (NumPy Compressed)
# ---------------------------------------------------------
# We save the IDs and the importances together so you can match them later
npz_path = explainer_save_dir / f'patient_importances_{DATA_NAME}_{TARGET_NAME}_{SAMPLE_SIZE}.npz'

# Convert mapping edges to a raw array for reference
edge_reference = np.array([[unique_nodes[src], unique_nodes[dst]] for src, dst in mapping_edges])

np.savez_compressed(
    npz_path,
    eids=np.array(eids_to_run),               # Patient IDs in exact order
    importances=all_importances_norm,         # Shape: (n_patients, n_edges)
    edge_names=edge_reference                 # The string literal names for each edge index
)
print(f"\nSaved raw patient-level importances to:\n{npz_path}")

In [ ]:
# ---------------------------------------------------------
# Generate the Plots
# ---------------------------------------------------------

# 1. Plot Top-K Selection Frequency
# Extract a subset of tuples: (source, target, frequency)
freq_edges = [(res['source'], res['target'], res['top_k_freq']) for res in global_results]

plot_global_importance_brain_map(
    edges_list=freq_edges, 
    title=f"Global Importance: Top {TOP_K} Frequency\nCohort Size: {n_patients} | Target: {TARGET_NAME}", 
    n_lines=TOP_K,
    colormap="Blues"  
)

# 2. Plot Mean Normalized Importance
# Extract a subset of tuples: (source, target, mean_importance)
mean_edges = [(res['source'], res['target'], res['mean_norm_imp']) for res in global_results]

plot_global_importance_brain_map(
    edges_list=mean_edges, 
    title=f"Global Importance: Mean Normalized Weight\nCohort Size: {n_patients} | Target: {TARGET_NAME}", 
    n_lines=TOP_K,
    colormap="Reds"   
)

In [ ]:
# 1. Load the saved CSV
csv_path = explainer_save_dir / f'global_importance_{DATA_NAME}_{TARGET_NAME}_{SAMPLE_SIZE}.csv'
saved_df = pd.read_csv(csv_path)

# 2. Extract the tuples directly (no manual sorting needed)
freq_edges = [(row.source, row.target, row.top_k_freq) for _, row in saved_df.iterrows()]
mean_edges = [(row.source, row.target, row.mean_norm_imp) for _, row in saved_df.iterrows()]

# 3. Plot them directly
plot_global_importance_brain_map(
    edges_list=freq_edges, 
    title=f"Global Importance: Top {TOP_K} Frequency\nCohort Size: {n_patients} | Target: {TARGET_NAME}", 
    n_lines=TOP_K,
    colormap="Blues"  
)

plot_global_importance_brain_map(
    edges_list=mean_edges, 
    title=f"Global Importance: Mean Normalized Weight\nCohort Size: {n_patients} | Target: {TARGET_NAME}", 
    n_lines=TOP_K,
    colormap="Reds"   
)